# Direct Corpus Interaction (DCI) — LLM Wiki

A tool-calling agent that answers questions from the compiled `wiki/` bundle.
The model never reads the corpus directly — it uses three tools (`list_pages`, `grep_wiki`, `read_page`)
injected as `role: tool` messages, following the standard ReAct loop.

**Prerequisites** — install from the project root:
```bash
uv sync           # creates .venv with the anthropic SDK
cp .env.example .env && nano .env   # paste ANTHROPIC_API_KEY
```

In [ ]:
"""
Cell 1 — Environment setup
Loads ANTHROPIC_API_KEY from .env (mirrors what start-claude.sh does for the CLI).
"""
import os
from pathlib import Path
from dotenv import load_dotenv          # included in the anthropic SDK's extras
                                        # or: pip install python-dotenv

# Walk up from the notebook to find the .env next to pyproject.toml
_root = Path(__file__).parent if "__file__" in dir() else Path.cwd()
while _root != _root.parent:
    if (_root / ".env").exists():
        load_dotenv(_root / ".env")
        print(f"✅ loaded .env from {_root / '.env'}")
        break
    _root = _root.parent
else:
    # Fallback: maybe the key is already in the environment (e.g. CI)
    if os.environ.get("ANTHROPIC_API_KEY"):
        print("✅ ANTHROPIC_API_KEY already set in environment.")
    else:
        raise EnvironmentError(
            "No .env found and ANTHROPIC_API_KEY is not set.\n"
            "Copy .env.example → .env and paste your key."
        )

In [ ]:
import json
import re
import sys
import time
from pathlib import Path

import anthropic

# ── locate project root (holds wiki/ and raw/) ────────────────────────────────
PROJECT_ROOT = Path.cwd()
for _c in [Path.cwd(), *Path.cwd().parents]:
    if (_c / "wiki").is_dir() and (_c / "raw").is_dir():
        PROJECT_ROOT = _c
        break

WIKI_ROOT   = PROJECT_ROOT / "wiki"
SCRIPT_DIR  = PROJECT_ROOT / ".agents" / "skills" / "llm-wiki" / "scripts"
sys.path.insert(0, str(SCRIPT_DIR))   # makes util and prompts importable

import util        # the existing stateless helpers
import prompts     # DCI_SYSTEM prompt lives here already

print(f"Project root : {PROJECT_ROOT}")
print(f"Wiki bundle  : {WIKI_ROOT}")
print(f"Pages found  : {sum(len(util.list_pages(str(WIKI_ROOT), k)) for k in util.PAGE_DIRS)}")

In [ ]:
# ── load every wiki page once; grep_wiki searches across all of them ──────────
PAGES: dict[str, str] = {}
for subdir in util.PAGE_DIRS:
    folder = WIKI_ROOT / subdir
    if folder.is_dir():
        for md in sorted(folder.glob("*.md")):
            page_id = f"{subdir}/{md.stem}"   # e.g. "entities/semantido"
            PAGES[page_id] = md.read_text(encoding="utf-8")

WIKI_INDEX = util.build_index(str(WIKI_ROOT))   # what list_pages() returns to the model

print(f"Loaded {len(PAGES)} pages into memory.")

In [ ]:
# ── three corpus tools ─────────────────────────────────────────────────────────

def list_pages_tool() -> str:
    """Return the full wiki index so the model can see every page ID."""
    return WIKI_INDEX


def grep_wiki(pattern: str) -> str:
    """Search a regex (or plain text) across all wiki pages with line provenance."""
    try:
        rx = re.compile(pattern, re.I)
    except re.error:
        rx = re.compile(re.escape(pattern), re.I)
    hits = []
    for page_id, text in PAGES.items():
        for lineno, line in enumerate(text.splitlines(), 1):
            if rx.search(line):
                hits.append(f"{page_id}:{lineno}: {line}")
    return "\n".join(hits[:40]) or "(no matches)"


def read_page(page: str, start: int = 0) -> str:
    """Read a wiki page by concept ID, with optional offset for pagination."""
    if page not in PAGES:
        return f"(unknown page: {page!r} — call list_pages to see valid IDs)"
    chunk     = PAGES[page][start : start + 12_000]
    remaining = max(0, len(PAGES[page]) - start - 12_000)
    suffix    = (
        f"\n\n[…{remaining} chars remaining — call read_page with start={start + 12_000}]"
        if remaining else ""
    )
    return chunk + suffix


# ── dispatch map used by the agent loop ───────────────────────────────────────
DCI_FUNCS = {
    "list_pages": list_pages_tool,
    "grep_wiki":  grep_wiki,
    "read_page":  read_page,
}
print("Tool functions registered:", list(DCI_FUNCS))

In [ ]:
# ── JSON schemas for the Anthropic messages API ───────────────────────────────

def _schema(name, description, properties, required=()):
    return {
        "name": name,
        "description": description,
        "input_schema": {
            "type": "object",
            "properties": properties,
            "required": list(required),
        },
    }


DCI_TOOLS = [
    _schema(
        "list_pages",
        "Read the wiki index to discover all page IDs and their one-line descriptions.",
        {},
    ),
    _schema(
        "grep_wiki",
        "Search exact text or a regex across all wiki pages; returns up to 40 matching lines "
        "with page-ID:line-number provenance. Use short lexical queries.",
        {"pattern": {"type": "string", "description": "Plain text or Python regex to search for."}},
        required=["pattern"],
    ),
    _schema(
        "read_page",
        "Read a complete wiki page by its concept ID (e.g. 'entities/semantido'). "
        "Use start= to paginate past the 12 000-char window.",
        {
            "page":  {"type": "string", "description": "Concept ID exactly as shown by list_pages."},
            "start": {"type": "integer", "description": "Byte offset to start from (default 0)."},
        },
        required=["page"],
    ),
]
print(f"Registered {len(DCI_TOOLS)} tool schemas.")

In [ ]:
# ── model and system prompt ───────────────────────────────────────────────────
MODEL  = os.environ.get("LLM_MODEL", "claude-opus-4-5")
CLIENT = anthropic.Anthropic()

# DCI_SYSTEM is already defined in prompts.py — we extend it with tool guidance
SYSTEM = prompts.DCI_SYSTEM + f"""

Available tools:
  • list_pages()                 — get the full wiki index (start here)
  • grep_wiki(pattern)           — fast text/regex search across all pages
  • read_page(page, start=0)     — read a page by concept ID; paginate with start=

Strategy:
  1. Call list_pages() once to orient yourself.
  2. grep_wiki() to locate the most relevant page IDs.
  3. read_page() the candidates; paginate if truncated.
  4. Synthesise and cite pages by concept ID, e.g. (entities/semantido).

This wiki covers: {WIKI_INDEX.splitlines()[0] if WIKI_INDEX else 'an OKF bundle'}.
"""
print(f"Model: {MODEL}")
print(f"System prompt: {len(SYSTEM)} chars")

In [ ]:
# ── agent loop ────────────────────────────────────────────────────────────────

def ask_model(messages: list, tools: list) -> anthropic.types.Message:
    """One chat turn — returns the full Message object (not just the first choice)."""
    return CLIENT.messages.create(
        model=MODEL,
        max_tokens=4096,
        system=SYSTEM,
        tools=tools,
        messages=messages,
    )


def run_agent(question: str, max_turns: int = 8) -> dict:
    messages = [{"role": "user", "content": question}]
    trace, evidence_chars = [], 0
    t0 = time.perf_counter()

    for turn in range(max_turns):
        response = ask_model(messages, DCI_TOOLS)

        # ── append the assistant turn ─────────────────────────────────────────
        messages.append({"role": "assistant", "content": response.content})

        # ── check stop reason ─────────────────────────────────────────────────
        if response.stop_reason == "end_turn":
            # collect text from all TextBlock items
            answer = " ".join(
                block.text for block in response.content
                if hasattr(block, "text")
            ).strip()
            return {
                "answer":         answer,
                "trace":          trace,
                "evidence_chars": evidence_chars,
                "latency_s":      round(time.perf_counter() - t0, 2),
                "stopped":        "answer",
                "turns":          turn + 1,
            }

        # ── dispatch tool calls ───────────────────────────────────────────────
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            name = block.name
            args = block.input or {}
            try:
                if name not in DCI_FUNCS:
                    raise KeyError(f"unknown tool {name!r}")
                result = str(DCI_FUNCS[name](**args))
            except Exception as exc:
                result = f"(tool error: {type(exc).__name__}: {exc})"
            evidence_chars += len(result)
            trace.append({"tool": name, "args": args, "chars": len(result)})
            tool_results.append({
                "type":        "tool_result",
                "tool_use_id": block.id,
                "content":     result,
            })

        messages.append({"role": "user", "content": tool_results})

    return {
        "answer":         "(max turns reached — try a more specific question or raise max_turns)",
        "trace":          trace,
        "evidence_chars": evidence_chars,
        "latency_s":      round(time.perf_counter() - t0, 2),
        "stopped":        "max_turns",
        "turns":          max_turns,
    }


print("Agent loop ready.")

In [ ]:
def show(result: dict) -> None:
    """Pretty-print a run_agent result."""
    print("=" * 70)
    print(result["answer"])
    print()
    print(f"  stopped={result['stopped']}  turns={result['turns']}"
          f"  latency={result['latency_s']}s  evidence={result['evidence_chars']:,} chars")
    if result["trace"]:
        print("\n  Tool trace:")
        for step in result["trace"]:
            args_str = ", ".join(f"{k}={v!r}" for k, v in step["args"].items())
            print(f"    → {step['tool']}({args_str})  [{step['chars']:,} chars returned]")
    print("=" * 70)

In [ ]:
result = run_agent("What does the wiki say about semantic layers and why they matter?")
show(result)

In [ ]:
# Try any of these:
queries = [
    "What is semantido and what problems does it solve?",
    "What are the key differences between text2sql and a semantic layer approach?",
    "Which entities are mentioned most often across sources?",
    "What open questions or contradictions exist in the wiki?",
]

for q in queries[:1]:          # change [:1] to [:4] to run all four
    print(f"\nQ: {q}")
    show(run_agent(q))

### 1. Ensure your wiki is compiled first
uv run python .agents/skills/llm-wiki/scripts/wiki.py ingest

### 2. Open the notebook
uv run jupyter lab dci.ipynb
### or open it in PyCharm (it supports Jupyter notebooks natively)

### 3. Run all cells top-to-bottom (Kernel → Restart & Run All)